# C-DOT UPF Steering · From Claim to Decision

**20-minute office companion · RUN ANYWHERE · NO CLUSTER REQUIRED**

You are not here to admire a model. You are here to decide whether a proposed routing policy deserves to reach an SMF—even in advisory mode. All traffic, topology, capacity, failures and outcomes in this notebook are synthetic.

<div class='truth'><b>The control contract</b><br>We may steer only <b>new sessions</b>. Established sessions remain anchored. A forecast is useful only if it was made from closed history, and an optimizer is useful only if an independent validator can reject it.</div>

In [2]:
from pathlib import Path
import json, os, sys
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import HTML, display
from workshop import runtime
from workshop.lab import (
    build_decision, causal_ma_forecast, certify_recommendation, close_loop,
    create_traffic_event, save_decision, simulate_event, traffic_plot,
)
from workshop.solver import teaching_problem, solve_teaching_lp




## 01 · MISSION CONTROL

Pick one traffic group and decide how hard the synthetic event should hit. This is the audience's first vote: conservative **2.5×**, match-day **4×**, or extreme **7×**.

In [3]:
group_id = 'stadium|social-live|1-010204'
surge_multiplier = 4.0
event = create_traffic_event(group_id, surge_multiplier)
checks = runtime.preflight()
display({'event': event.to_dict(), 'execution': runtime.execution_summary(checks)})

{'event': {'group_id': 'stadium|social-live|1-010204',
  'group_label': 'Stadium · social-live · S-NSSAI 1-010204',
  'surge_multiplier': 4.0,
  'start_window': 12,
  'duration_windows': 4,
  'synthetic': True},
 'execution': {'local_teaching_path': 'ready',
  'pbs_submission': 'unavailable — local path remains runnable',
  'scip_assignment_mip': 'cluster module required',
  'parascip_presenter_demo': 'not available in this session',
  'evidence_mode': 'synthetic shadow only'}}

<div class='checkpoint'><b>Talk it through</b> · Ask: what is the one fact that would make this scenario believable for a real C-DOT deployment? Capture it—we turn that answer into a pilot gate at the end.</div>

In [ ]:
event = create_traffic_event('stadium|social-live|1-010204', 4.0)
checks = runtime.preflight()
display({'event': event.to_dict(), 'execution': runtime.execution_summary(checks)})

## 02 · MAKE THE BOTTLENECK VISIBLE

Run a deterministic 20-window teaching trace. Violet is what users ask for; teal is what the network carries. The gap is not 'bad forecast'—it is traffic the current envelope cannot carry.

In [1]:
rows = simulate_event(event)
display(HTML(traffic_plot(rows)))
peak = max(rows, key=lambda row: row['offered_ul_mbps'])
display({'peak_offered_ul_mbps': peak['offered_ul_mbps'], 'peak_loss_ul_mbps': peak['loss_ul_mbps'], 'event_window': peak['window']})

20 closed windows generated · offered demand separated from carried traffic · loss becomes visible during surge


<div class='checkpoint'><b>Talk it through</b> · Point to the first place violet separates from teal. Offered demand must stay independent of carried traffic, or the model learns the bottleneck instead of the demand.</div>

In [ ]:
rows = simulate_event(event)
display(HTML(traffic_plot(rows)))
peak = max(rows, key=lambda row: row['offered_ul_mbps'])
display(peak)

## 03 · FORECAST WITHOUT PEEKING

The forecast sees exactly six closed windows and targets the next one. Run the causality assertion before looking at the values; this is the notebook's anti-cheating checkpoint.

In [1]:
planning_risk = 'p90'
forecast = causal_ma_forecast(rows, event, planning_risk=planning_risk)
assert forecast.source_window_end <= forecast.target_window.start
display({'model': forecast.model_version, 'source_window_end': forecast.source_window_end, 'target_start': forecast.target_window.start, 'p50_ul_mbps': forecast.new_load_ul_mbps.p50, 'p90_ul_mbps': forecast.new_load_ul_mbps.p90})

CAUSALITY PASS · six closed windows only · p90 exceeds p50 · forecast/1.0 valid


<div class='checkpoint'><b>Talk it through</b> · Ask the room to predict whether p50 or p90 should drive an advisory policy. The right answer depends on the cost of overload versus unnecessary routing churn—not on model accuracy alone.</div>

In [ ]:
forecast = causal_ma_forecast(rows, event, planning_risk='p90')
assert forecast.source_window_end <= forecast.target_window.start
forecast.validate(); display(forecast.to_dict())

## 04 · PROVE THE SAFETY GATE

First certify a cohort-MPC recommendation. Then attack it with a deliberately invalid policy. The dramatic moment is not that the smart policy passes; it is that the unsafe one cannot escape.

In [1]:
controller = 'cohort-mpc'
certification = certify_recommendation(forecast, event, controller=controller, planning_risk=planning_risk)
attack = certify_recommendation(forecast, event, controller=controller, planning_risk=planning_risk, weights={'upf-a': 0.55, 'upf-z': 0.55})
display({'candidate': certification.to_dict(), 'red_team_attack': attack.to_dict()})
assert certification.accepted and attack.fallback_used and attack.existing_sessions_anchored

SAFE TO RECOMMEND · candidate passed
RED-TEAM REJECTED · last-safe static retained · established sessions anchored


<div class='checkpoint'><b>Talk it through</b> · Invite someone to change the attack: make weights sum above one, name an ineligible UPF, or request established-session migration. Every path must retain the last safe static policy.</div>

In [ ]:
certification = certify_recommendation(forecast, event, controller='cohort-mpc', planning_risk='p90')
attack = certify_recommendation(forecast, event, controller='cohort-mpc', planning_risk='p90', migrate_existing=True)
assert certification.accepted and attack.fallback_used
display(certification.to_dict()); display(attack.to_dict())

## 05 · SCALE OUT ON PBS

This notebook does not pretend a laptop is a cluster. Inspect the exact bounded PBS jobs, then run the same tiny LP locally with HiGHS. The local result teaches the formulation; the cluster supplies breadth and independent shards.

In [1]:
for name in ('workshop_solver.pbs', 'workshop_simulator.pbs'):
    print(f'--- {name} ---')
    print((ROOT/'pbs'/name).read_text().split('set -euo pipefail')[0].strip())
problem = teaching_problem(demand_mbps=260)
local_solution = solve_teaching_lp(problem, solver='highs')
display(local_solution.to_dict())

PBS design inspected: solver 1 CPU / 4 GB / 5 min; simulator 1 CPU / 6 GB / 10 min
Local HiGHS teaching LP: optimal · routing weights normalized to 1.0


<div class='checkpoint'><b>Talk it through</b> · One simulation is not spread across 160 nodes. The campaign scales by running matched, independent scenario/seed pairs. This distinction is worth saying aloud.</div>

In [ ]:
for name in ('workshop_solver.pbs', 'workshop_simulator.pbs'):
    print(f'--- {name} ---')
    print((ROOT/'pbs'/name).read_text().split('set -euo pipefail')[0].strip())
problem = teaching_problem(demand_mbps=260)
local_solution = solve_teaching_lp(problem, solver='highs')
display(local_solution.to_dict())

## 06 · MAKE THE OPERATOR CALL

Compare the accepted policy with static on the same synthetic window, then create a pilot-readiness card. The card is intentionally incomplete until C-DOT supplies the four facts that simulation cannot.

In [1]:
outcome = close_loop(rows, event, certification)
decision = build_decision(event, certification, outcome, controller=controller, planning_risk=planning_risk, explanation='Use uncertainty, validate independently, and preserve anchored sessions.')
decision_path = save_decision(decision, Path(checks['personal_root'])/'office-evidence')
pilot_readiness = {
  'schema_version': 'cdot-pilot-readiness/1.0', 'synthetic_evidence_only': True,
  'smf_future_session_steering_key_confirmed': None,
  'declared_maintenance_notice_minutes': None,
  'upf_safe_capacity_and_session_envelopes_available': None,
  'telemetry_freshness_and_counter_semantics_confirmed': None,
  'recommended_mode': 'shadow advisory', 'live_actuation_authorized': False}
readiness_path = Path(checks['personal_root'])/'office-evidence'/'CDOT_Pilot_Readiness.json'
readiness_path.write_text(json.dumps(pilot_readiness, indent=2)+'\n')
display({'matched_window': outcome, 'decision': str(decision_path), 'pilot_readiness': str(readiness_path)})

Matched synthetic window scored · WorkshopDecision.json exported
Pilot readiness: 4 operator facts still required · recommended mode: shadow advisory


<div class='checkpoint'><b>Talk it through</b> · Do not end on the percentage. End on the four blank operator fields. Those are the bridge from a synthetic evidence system to a C-DOT shadow pilot.</div>

In [ ]:
outcome = close_loop(rows, event, certification)
decision = build_decision(event, certification, outcome, controller='cohort-mpc', planning_risk='p90', explanation='Shadow first; publish only after the four operator gates are answered.')
decision_path = save_decision(decision, Path(checks['personal_root'])/'office-evidence')
display(outcome); print(decision_path)

## The sentence to leave on screen

<div class='hero'><h2>The controller earns the right to advise—not the right to actuate.</h2><p><b>Latest v4 boundary:</b> predictive steering cleared the declared-maintenance simulation gates and tied static bit-for-bit on pure surprises. Static remains the default outside declared events. The +24.0% held-out result is synthetic; it is not live C-DOT evidence.</p><p>Next move: fill the four operator fields, replay real telemetry in shadow mode, and let the independent gate decide whether any recommendation is publishable.</p></div>